# Preprocessing: UCI Bank Marketing Dataset

**CAP182 SS2 Capstone – STADIO Equities First-Deposit Activation** 

**Dataset:** UCI Bank Marketing Dataset (`bank-full.csv`) (Moro, Laureano & Cortez, 2011)

---

## Introduction

This notebook focuses on preparing and cleaning the data before any machine learning techniques are applied. Data preprocessing is an important step in the data science process because it helps ensure that the dataset is accurate, consistent and suitable for analysis.

The notebook only covers data preparation activities and does not include model building, training or evaluation. These tasks will be completed in later stages of the project, in line with the data science pipeline.

The data science workflow followed in this project is:

`Data Preparation → Algorithm Selection → Model Building → Training/Fitting → Evaluation → Re-evaluation`

The aim of this stage is to explore the dataset, identify any data quality issues, and prepare the variables so that they can be used effectively in later machine learning tasks.
`

## 1. Purpose and Project Context

STADIOEquities is a South African digital investment platform. About 41% of its 2.3 million registered accounts have never received a deposit. This creates an activation gap, as accounts that remain unfunded generate little or no revenue. The central business problem is therefore predicting which registered customers are unlikely to make their first deposit. By identifying the characteristics associated with activation, the company can target customers more effectively and improve conversion from registered accounts to active investors.

Because STADIOEquities' proprietary customer data are unavailable, the UCI Bank Marketing dataset is used as a proxy to demonstrate the modelling approach. Its binary target (y) records whether a contacted customer subscribed to a term deposit. This differs from STADIOEquities' first-deposit activation outcome and therefore cannot directly identify STADIOEquities customers who will remain unfunded. Instead, the dataset is used to demonstrate how customer characteristics and behavioural/contact variables can be analysed to distinguish between customers who do and do not complete a defined financial conversion outcome.

Although the two contexts are not identical, they share a similar objective: identifying factors associated with whether a customer completes a financial conversion. For this project, the target variable y (whether a client subscribed to a term deposit) is used as a proxy for first-deposit activation.

The dataset is not used to draw conclusions about STADIOEquities. Instead, it demonstrates the data science process for the CAP182 project, including data preparation, feature engineering, model development and evaluation.

The aim of this notebook is to prepare the dataset for analysis by identifying data-quality issues, examining the available variables and producing a clean dataset for the next stage.

## 2. Dataset Loading

**2.1. Project Structure and Working Environment**

The project directory was organised according to the main stages of the data science workflow required for this capstone project.

The project contains separate folders for raw and processed data, preprocessing, feature engineering, model development and model evaluation. This structure separates the stages of the analysis and allows the output from one stage to be used as the input for the next stage.

The raw dataset will be stored in `data/raw/`. The preprocessing stage will produce a cleaned dataset in `data/processed/`, which will then be used for feature engineering and subsequent model development and evaluation.

The structure also supports reproducibility by using consistent project-relative file paths rather than machine-specific file locations.

In [2]:
import os

print(os.getcwd())

/home/jovyan/work/CAP182 - SS2


In [4]:
from pathlib import Path

project = Path("/home/jovyan/work/CAP182 - SS2")

folders = [
    "data/raw",
    "data/processed",
    "preprocessing",
    "feature_engineering",
    "models",
    "evaluation",
]

for folder in folders:
    (project / folder).mkdir(parents=True, exist_ok=True)

print("Project folders created successfully.")

Project folders created successfully.


In [5]:
for item in sorted(project.rglob("*")):
    if item.is_dir():
        print("📁", item.relative_to(project))

📁 .ipynb_checkpoints
📁 data
📁 data/processed
📁 data/raw
📁 evaluation
📁 feature_engineering
📁 models
📁 preprocessing


**2.2 Loading the Data**

The dataset was loaded from the `bank-full.csv` file. Before importing the data, the file structure was inspected to ensure that the data could be read correctly. This showed that the variables were separated by semicolons (`;`) rather than commas, so the appropriate delimiter was specified when loading the dataset. Verifying the file format before importing the data helped to ensure that all columns were imported accurately and reduced the risk of data quality issues during later stages of the analysis.

In [14]:
raw_file = Path("data/raw/bank-full.csv")

print("File exists:", raw_file.exists())
print("File location:", raw_file.resolve())

File exists: True
File location: /home/jovyan/work/CAP182 - SS2/data/raw/bank-full.csv


In [15]:
import pandas as pd

RAW_PATH = "data/raw/bank-full.csv"

df = pd.read_csv(RAW_PATH)

print("Dataset loaded successfully.")

Dataset loaded successfully.


## 3 Initial Dataset Inspection

An initial inspection is performed to establish the size and structure of the dataset before preprocessing. This includes examining the number of observations and variables.

Understanding the dataset structure provides a baseline for identifying data-quality issues and determining the appropriate preprocessing steps.

In [16]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Number of rows: 45211
Number of columns: 1


The initial inspection shows 45,211 observations but only one column. This indicates that the delimiter used in the source CSV has not been interpreted correctly during loading.

The dataset uses a semicolon (`;`) as its field separator. The dataset will therefore be reloaded using the correct delimiter before further inspection or preprocessing.

In [17]:
RAW_PATH = "data/raw/bank-full.csv"

df = pd.read_csv(RAW_PATH, sep=";")

print("Dataset loaded successfully.")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Dataset loaded successfully.
Number of rows: 45211
Number of columns: 17


**3.1 Inspecting the Dataset Variables**

The variables in the dataset are inspected to understand the available customer, campaign and outcome information.

This step helps distinguish numerical and categorical variables and provides the basis for selecting appropriate preprocessing techniques.

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  object
 16  y          45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB


The dataset contains 45,211 observations and 17 variables. Seven variables are stored as integers and ten are categorical variables stored as objects.

No standard null values are reported by the initial inspection, as all variables contain 45,211 non-null observations.

The target variable `y` is categorical and represents whether the customer subscribed to a term deposit. It will be used as the proxy outcome for first-deposit activation in this project.

**3.2 Checking for Duplicate Records**

The dataset does not contain a unique customer identifier, so duplicate customers cannot be identified directly.

Instead, an exact duplicate check is performed to identify records where all available variable values are identical. This provides a basic data-quality check before further preprocessing.

In [20]:
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 0


No exact duplicate rows were identified in the dataset. This means that no two records contain identical values across all 17 variables.

Because the dataset does not contain a unique customer identifier, this result should not be interpreted as confirmation that each record represents a unique customer.

**3.3 Checking for Unknown Categorical Values**

Although the dataset contains no standard null values, categorical variables may contain `"unknown"` values representing unavailable or unspecified information.

The frequency of `"unknown"` values is therefore examined before deciding how these observations should be handled during preprocessing.

In [21]:
categorical_columns = df.select_dtypes(include="object").columns

unknown_counts = (df[categorical_columns] == "unknown").sum()

print(unknown_counts[unknown_counts > 0])

job            288
education     1857
contact      13020
poutcome     36959
dtype: int64


**Unknown Value Assessment**

The dataset contains `"unknown"` values in four categorical variables: `job`, `education`, `contact` and `poutcome`.

The proportion of `"unknown"` values is particularly high for `poutcome`. These observations will not be removed because doing so would result in substantial data loss. Instead, `"unknown"` will initially be retained as a separate category so that the information contained in the original dataset is preserved.

The treatment of categorical variables will be addressed during feature engineering and model preparation.

**3.4 Examining the Target Variable**

The distribution of the target variable `y` is examined to determine the proportion of observations in each outcome class.

This is important for the project because the modelling task is a binary classification problem, and an uneven distribution between the two classes can affect model evaluation.

In [22]:
target_counts = df["y"].value_counts()

print(target_counts)
print("\nProportions:")
print(df["y"].value_counts(normalize=True))

y
no     39922
yes     5289
Name: count, dtype: int64

Proportions:
y
no     0.883015
yes    0.116985
Name: proportion, dtype: float64


**Target Distribution Assessment**

The target variable is imbalanced. Of the 45,211 observations, 39,922 (88.3%) have a `no` outcome and 5,289 (11.7%) have a `yes` outcome.

This imbalance means that accuracy alone may provide a misleading indication of model performance. A model that predominantly predicts the majority class could achieve high accuracy while performing poorly at identifying customers in the minority class.

Model evaluation will therefore consider additional classification metrics, including precision, recall and F1-score, together with the confusion matrix.

**3.5 Inspecting Numerical Variables**

The numerical variables are summarised to identify their ranges and potential unusual values before preprocessing.

This provides a basic data-quality check and helps determine whether any variables require special treatment during feature engineering.

In [28]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
age,45211.0,40.936210,10.618762,18.0,33.0,39.0,48.0,95.0
balance,45211.0,1362.272058,3044.765829,-8019.0,72.0,448.0,1428.0,102127.0
day,45211.0,15.806419,8.322476,1.0,8.0,16.0,21.0,31.0
duration,45211.0,258.163080,257.527812,0.0,103.0,180.0,319.0,4918.0
campaign,45211.0,2.763841,3.098021,1.0,1.0,2.0,3.0,63.0
pdays,45211.0,40.197828,100.128746,-1.0,-1.0,-1.0,-1.0,871.0
previous,45211.0,0.580323,2.303441,0.0,0.0,0.0,0.0,275.0


The numerical variables were examined using descriptive statistics to identify their ranges and potential data-quality considerations.

Most variables fall within plausible ranges. However, `pdays` contains a value of `-1`, which represents a special coded condition rather than a conventional numerical value and will require appropriate treatment during feature engineering.

The `duration` variable also requires particular consideration because it represents the duration of the last contact. If the intended prediction is made before or at the point of intervention, using information from a completed contact could introduce target leakage. This variable will therefore be assessed during feature engineering before the final modelling feature set is defined.

Variables such as `balance`, `campaign` and `previous` show substantial ranges or skewness. These values will not be removed solely because they are extreme; their usefulness and treatment will be considered during feature engineering and model preparation.

**Inspecting Categorical Variables**

The unique categories of the categorical variables are examined to identify the values present in the dataset and confirm that they are suitable for subsequent encoding.

In [30]:
categorical_columns = df.select_dtypes(include="object").columns

for column in categorical_columns:
    print(f"\n{column}:")
    print(df[column].unique())


job:
['management' 'technician' 'entrepreneur' 'blue-collar' 'unknown'
 'retired' 'admin.' 'services' 'self-employed' 'unemployed' 'housemaid'
 'student']

marital:
['married' 'single' 'divorced']

education:
['tertiary' 'secondary' 'unknown' 'primary']

default:
['no' 'yes']

housing:
['yes' 'no']

loan:
['no' 'yes']

contact:
['unknown' 'cellular' 'telephone']

month:
['may' 'jun' 'jul' 'aug' 'oct' 'nov' 'dec' 'jan' 'feb' 'mar' 'apr' 'sep']

poutcome:
['unknown' 'failure' 'other' 'success']

y:
['no' 'yes']


**Inspection Summary**

The initial inspection established that the dataset contains 45,211 observations and 17 variables, consisting of numerical and categorical data.

No standard null values or exact duplicate records were identified. Several categorical variables contain `"unknown"` values, which will be retained rather than removed because they represent unavailable or unspecified information in the original dataset.

The target variable `y` is binary and imbalanced, with the `no` outcome representing 88.3% of observations and the `yes` outcome representing 11.7%. This will be considered when selecting and evaluating the classification models.

The numerical-variable inspection identified two variables requiring specific consideration during subsequent preprocessing and feature engineering: `pdays`, which uses `-1` as a special coded value, and `duration`, which may introduce information leakage depending on the intended prediction point.

The dataset is therefore sufficiently understood to proceed to the preprocessing decisions.

## 4. Preprocessing Decisions

**4.1 Treatment of `pdays`**

The `pdays` variable records the number of days since the customer was previously contacted. A value of `-1` is used in the dataset to indicate that the customer was not previously contacted.

Because `-1` represents a categorical condition rather than a meaningful number of days, it should not be treated as an ordinary numerical value. The frequency of this coded value is examined before determining the appropriate feature representation.

In [32]:
pdays_unknown = (df["pdays"] == -1).sum()
pdays_unknown_pct = (df["pdays"] == -1).mean() * 100

print("pdays = -1:", pdays_unknown)
print("Percentage:", round(pdays_unknown_pct, 2), "%")

pdays = -1: 36954
Percentage: 81.74 %


**`pdays` Decision**

The inspection found that 36,954 observations (81.74%) have a `pdays` value of `-1`, indicating that the customer had not previously been contacted.

Because `-1` represents a condition rather than an actual number of days, it should not be interpreted as an ordinary numerical value.

The `pdays` variable will therefore not be used in its raw form as a continuous numerical variable. During feature engineering, a `previously_contacted` indicator will be derived to distinguish customers who had previously been contacted from those who had not. The actual number of days since previous contact will be considered separately for customers with a previous contact history.

**4.2 Treatment of `duration`**

The `duration` variable represents the duration of the customer's last contact with the campaign. Although the variable may contain predictive information, its value is only known after the contact has taken place.

Including `duration` in a model intended to identify customers before or during an activation intervention could therefore introduce target leakage, because it provides information that would not necessarily be available at the intended prediction point.

For this reason, `duration` will be excluded from the primary modelling feature set. This allows the models to focus on customer and campaign characteristics that could be available before the outcome is observed.

**4.3 Treatment of Categorical Variables**
    
The dataset contains several categorical variables that are stored as text values. These variables cannot be directly used as numerical inputs by the classification models.

One-hot encoding will therefore be used to represent categorical variables as binary indicator variables. This avoids imposing an artificial numerical ordering on categories.

The original categorical values will be retained during the preprocessing and feature-engineering stages. The encoding will be applied as part of the modelling pipeline so that the encoder is fitted using the training data only. This helps prevent information from the test data being incorporated into the model during training.

**4.4 Treatment of the Target Variable**

The target variable `y` represents whether the customer subscribed to a term deposit. For binary classification, the target will be represented numerically, with `no` coded as 0 and `yes` coded as 1.

In this project, this outcome is used as a proxy for the STADIOEquities first-deposit activation outcome. The target transformation does not imply that the underlying dataset represents STADIOEquities customers.

## 5. Applying Preprocessing

The preprocessing decisions identified above are now applied to a copy of the raw dataset. The original raw dataset is not modified so that the source data remains preserved and reproducible.

Only transformations required for the subsequent feature-engineering and modelling stages are applied at this stage.

In [34]:
# Create a copy of the raw dataset for preprocessing
df_processed = df.copy()

print("Processed DataFrame created.")
print("Rows:", df_processed.shape[0])
print("Columns:", df_processed.shape[1])

Processed DataFrame created.
Rows: 45211
Columns: 17


**5.1 Handling the `pdays` Special Value**

The value `-1` in `pdays` represents customers who were not previously contacted rather than an actual number of days.

For preprocessing, these values are converted to missing values (`NaN`) so that `-1` is not interpreted as a meaningful numerical distance. The information about whether a customer was previously contacted will be represented explicitly during the feature-engineering stage.

In [35]:
import numpy as np

df_processed["pdays"] = df_processed["pdays"].replace(-1, np.nan)

print("Remaining -1 values:", (df_processed["pdays"] == -1).sum())
print("Missing pdays values:", df_processed["pdays"].isna().sum())

Remaining -1 values: 0
Missing pdays values: 36954


The `-1` values in `pdays` were successfully converted to missing values, with 36,954 observations affected. No rows were removed.

The resulting missing values will be handled through feature engineering, where the information that a customer had no previous contact will be represented explicitly.

**5.2 Encoding the Target Variable**

The target variable `y` is currently represented using the categorical values `no` and `yes`.

For binary classification, these outcomes are converted to numerical values, with `no` represented as 0 and `yes` represented as 1. This creates a binary target suitable for subsequent model development.

In [36]:
df_processed["y_binary"] = df_processed["y"].map({
    "no": 0,
    "yes": 1
})

print(df_processed["y_binary"].value_counts())

y_binary
0    39922
1     5289
Name: count, dtype: int64


**Target Encoding Result**

The target variable was successfully converted from categorical values to a binary numerical representation. The class distribution remains unchanged, with 39,922 observations coded as 0 and 5,289 observations coded as 1.

The original `y` variable has been retained at this stage so that the original target representation remains available for reference.

**5.3 Preprocessing Validation**

The processed dataset is checked to confirm that the expected transformations were applied without changing the number of observations.

In [37]:
print("Rows:", df_processed.shape[0])
print("Columns:", df_processed.shape[1])
print("Missing pdays:", df_processed["pdays"].isna().sum())
print("Unique y_binary values:", df_processed["y_binary"].unique())

Rows: 45211
Columns: 18
Missing pdays: 36954
Unique y_binary values: [0 1]


The validation confirms that the preprocessing steps were applied without changing the number of observations. The `pdays` special value has been converted to missing values, and the new `y_binary` variable contains only the expected binary values of 0 and 1.

The original `y` variable has been retained, and no rows have been removed during preprocessing.

## 6. Saving the Preprocessed Dataset

The processed dataset is saved separately from the raw dataset so that the original data remains unchanged.

The resulting file will be used as the input for the feature-engineering stage of the project.

In [39]:
PROCESSED_PATH = "data/processed/bank_marketing_preprocessed.csv"

df_processed.to_csv(PROCESSED_PATH, index=False)

print("Processed dataset saved successfully.")
print("Location:", PROCESSED_PATH)

Processed dataset saved successfully.
Location: data/processed/bank_marketing_preprocessed.csv
